In [ ]:
!pip uninstall -y m2stitch
!pip install m2stitch

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams["font.family"] = "sans-serif"
from skimage import io
from pathlib import Path
import m2stitch

In [ ]:
datadir = Path("/Users/fukai/Desktop/drive-download-20231226T035837Z-001/tiling2/")

In [ ]:
files = list(datadir.glob("*.tif"))
poss = []
images = []
for f in files:
    images.append(io.imread(str(f)))
    poss.append(f.stem.split("-")[1].split("_"))
poss = np.array(poss, dtype=np.int32)
images = np.array(images)
print(images.shape)
print(poss)

In [ ]:
tiled_image = np.zeros(
    ((np.max(poss[:, 0]) + 1) * images.shape[1], 
     (np.max(poss[:, 1]) + 1) * images.shape[2])
)
for i, (x, y) in enumerate(poss):
    tiled_image[x*images.shape[1]:(x+1)*images.shape[1], 
                y*images.shape[2]:(y+1)*images.shape[2]] = images[i]
plt.imshow(tiled_image)

In [ ]:
overlap_percentage = 10 # change this value according to your imaging condition
position_initial_guess = poss*images.shape[1:]*(100-overlap_percentage)/100
position_initial_guess

In [ ]:
tiled_image = np.zeros(
    ((np.max(poss[:, 0]) + 1) * images.shape[1], 
     (np.max(poss[:, 1]) + 1) * images.shape[2])
)
for i, (x, y) in enumerate(position_initial_guess.astype(np.int32)):
    tiled_image[x:x+images.shape[1], y:y+images.shape[2]] += images[i]
plt.imshow(tiled_image)

In [ ]:
result_df, props = m2stitch.stitch_images(
    images, 
    position_indices=poss, 
    position_initial_guess=position_initial_guess,
    row_col_transpose=False,
    full_output=True,
    ncc_threshold=0.1,
    #outlier_predictor_name="none",
)

In [ ]:
# stitching example
result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()

size_y = images.shape[1]
size_x = images.shape[2]

stitched_image_size = (
    result_df["y_pos2"].max() + size_y,
    result_df["x_pos2"].max() + size_x,
)
stitched_image = np.zeros_like(images, shape=stitched_image_size)
for i, row in result_df.iterrows():
    #break
    #if row["col"] == 2 and row["row"] == 2:
    #    continue
    stitched_image[
        row["y_pos2"] : row["y_pos2"] + size_y,
        row["x_pos2"] : row["x_pos2"] + size_x,
    ] = images[i]

plt.imshow(stitched_image)